In [2]:
import os
import sys
from dotenv import load_dotenv
 
# load_dotenv(".env")
load_dotenv("../.env")
GITHUB_MCP_URL = os.getenv("GITHUB_MCP_URL")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
 
if not GITHUB_MCP_URL:
    raise ValueError("GITHUB_MCP_URL is missing from .env")
 
if not GITHUB_TOKEN or GITHUB_TOKEN == "your_token_goes_here":
    raise ValueError("Set your GitHub token in .env before continuing")
 
print("Endpoint configured:", GITHUB_MCP_URL)
print("GitHub token loaded:", bool(GITHUB_TOKEN))

Endpoint configured: https://api.githubcopilot.com/mcp/
GitHub token loaded: True


In [3]:
# Import
sys.path.insert(0, "src")
from src.mcp_clients import GitHubMCPClient
from src.mcp_clients import Context7MCPClient

In [4]:
# connect and discover tool
async with GitHubMCPClient() as github:
    github_tools = await github.list_tools()

    for tool in github_tools:
        print(tool.name, "-", tool.description)

add_comment_to_pending_review - Add review comment to the requester's latest pending pull request review. A pending review needs to already exist to call this (check with the user if not sure).
add_issue_comment - Add a comment and/or reaction to a specific issue or issue comment in a GitHub repository. Use this tool with pull requests as well (in this case pass pull request number as issue_number), but only if user is not asking specifically to add or react to review comments. At least one of body or reaction is required.
add_reply_to_pull_request_comment - Add a reply and/or reaction to an existing pull request comment. This can create a new comment linked as a reply to the specified comment, add an emoji reaction to the specified comment, or do both. At least one of body or reaction is required.
create_branch - Create a new branch in a GitHub repository
create_or_update_file - Create or update a single file in a GitHub repository. 
If updating, you should provide the SHA of the file

In [4]:
# call github tool
async with GitHubMCPClient() as github:
    await github.list_tools()
 
    # Replace these placeholders using the discovered schema.
    result = await github.call_tool(
        name="create_repository",
        arguments={
            "name":"my-repo",
            "description": "Created via MCP",
            "private": True,
            "auto-init": True
        },
    )
 
    print(result)

meta=None content=[TextContent(type='text', text='failed to create repository: Repository creation failed.\nRepository.name (custom): name already exists on this account', annotations=None, meta=None)] structured_content=None is_error=True result_type='complete'


In [5]:
async with GitHubMCPClient() as github:
    await github.list_tools()

    result = await github.call_tool(
        name="create_or_update_file",
        arguments={
            "owner": "aashwini1010",
            "repo": "my-repo",
            "branch": "main",
            "path": "hello.py",
            "message": "Add hello.py",
            "content": "print('Hello from GitHub MCP!')"
        },
    )

    print(result)

meta=None content=[TextContent(type='text', text='File already exists at hello.py. You must provide the current file\'s SHA when updating. Call get_file_contents with owner="aashwini1010", repo="my-repo", path="hello.py", and ref="main" to read the file you are about to overwrite; its first text result reports the blob SHA for the requested path. Then retry with the sha parameter set to the blob SHA that call reports.', annotations=None, meta=None)] structured_content=None is_error=True result_type='complete'


In [6]:
async with GitHubMCPClient() as github:
    await github.list_tools()

    result = await github.call_tool(
        name="get_file_contents",
        arguments={
            "owner": "aashwini1010",
            "repo": "my-repo",
            "path": "hello.py"
        },
    )

    print(result)

meta=None content=[TextContent(type='text', text='successfully downloaded text file (SHA: 7cf22b37b52548de161cc8a4456a6c1efe24465d)', annotations=None, meta=None), EmbeddedResource(type='resource', resource=TextResourceContents(uri='repo://aashwini1010/my-repo/sha/1683c5c151616b60d98762cf92ef1c5aabfa2d0c/contents/hello.py', mime_type='text/plain; charset=utf-8', meta=None, text="print('Hello from GitHub MCP!')"), annotations=None, meta=None)] structured_content=None is_error=False result_type='complete'


In [7]:
async with GitHubMCPClient() as github:
    await github.list_tools()

    result = await github.call_tool(
        name="search_users",
        arguments={
            "query": "aashwini1010"
        },
    )

    print(result)

meta=None content=[TextContent(type='text', text='{"total_count":1,"incomplete_results":false,"items":[{"login":"aashwini1010","id":335644774,"profile_url":"https://github.com/aashwini1010","avatar_url":"https://avatars.githubusercontent.com/u/335644774?v=4"}]}', annotations=None, meta=None)] structured_content=None is_error=False result_type='complete'


In [8]:
async with GitHubMCPClient() as github:
    await github.list_tools()

    result = await github.call_tool(
        name="search_code",
        arguments={
            "query": "hello.py"
        },
    )

    print(result)


meta=None content=[TextContent(type='text', text='{"total_count":160768,"incomplete_results":false,"items":[{"name":"git.org","path":"git.org","sha":"7ac29605cb07c3036c549b5e6d4a252223e41ddc","repository":"darshanime/notes","text_matches":[{"object_url":"https://api.github.com/repositories/93383943/contents/git.org?ref=356f769267b234c8714b42ca4ca8adc0aa58f5e3","object_type":"FileContent","property":"content","fragment":"The ~ character is useful for making relative references to the parent of a commit. For example, 3157e~1 refers to the commit before 3157e, and HEAD~3 is the great-grandparent of the current commit.\\n\\n\\ngit log --author=\\"John Smith\\" -p hello.py\\nThis will display a full diff of all the changes John Smith has made to the file hello.py.\\n\\nThe .. syntax is a very useful tool for comparing branches.","matches":[{"text":"hello.py","indices":[234,242]},{"text":"hello.py","indices":[324,332]}]}]},{"name":"README.md","path":"README.md","sha":"8814f465fc610a4c84625c6

In [9]:
# conect to context7
async with Context7MCPClient() as context7:
    context7_tools = await context7.list_tools()

    for tool in context7_tools:
        print(tool.name, "-", tool.description)


resolve-library-id - Resolves a package/product name to a Context7-compatible library ID and returns matching libraries.

You MUST call this function before 'Query Documentation' tool to obtain a valid Context7-compatible library ID UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.

Each result includes:
- Library ID: Context7-compatible identifier (format: /org/project)
- Name: Library or package name
- Description: Short summary
- Code Snippets: Number of available code examples
- Source Reputation: Authority indicator (High, Medium, Low, or Unknown)
- Benchmark Score: Quality indicator (100 is the highest score)
- Versions: List of versions if available. Use one of those versions if the user provides a version in their query. The format of the version is /org/project/version.

For best results, select libraries based on name match, source reputation, snippet coverage, benchmark score, and relevance to your use cas

In [10]:
async with Context7MCPClient() as context7:
    await context7.list_tools()
 
    # Replace these placeholders using the discovered schema.
    result = await context7.call_tool(
        name="resolve-library-id",
        arguments={
            "libraryName": "next.js",
            "query": "How do i setup middleware to check for valid JWT cookie?"
        },
    )
 
    print(result)

meta=None content=[TextContent(type='text', text='Available Libraries:\n\n- Title: Next.js\n- Context7-compatible library ID: /vercel/next.js\n- Description: Next.js enables you to create full-stack web applications by extending the latest React features and integrating powerful Rust-based JavaScript tooling for the fastest builds.\n- Code Snippets: 5896\n- Source Reputation: High\n- Benchmark Score: 82.01\n- Versions: v14.3.0-canary.87, v13.5.11, v15.1.8, v15.4.0-canary.82, v12.3.7, v11.1.3, v16.0.3, __branch__01-02-copy_58398, v16.1.0, v16.1.1, v15.1.11, v16.1.5, v16.1.6, v16.2.2, __branch__15-6-0-canary-57, v16.2.9\n----------\n- Title: Next.js\n- Context7-compatible library ID: /websites/nextjs\n- Description: Next.js is a React framework for building full-stack web applications. It provides additional features and optimizations, automatically configuring lower-level tools to help developers focus on building products quickly.\n- Code Snippets: 6876\n- Source Reputation: High\n- Be